# Fine-tune FunctionGemma 270M on Jarvis-CD

Adapted from Unsloth's official `FunctionGemma_(270M).ipynb`.

- Base model: `unsloth/functiongemma-270m-it`
- Dataset: `v7_2k/jarvis_v7_functiongemma.jsonl` (1,979 examples: 589 single / 590 multi / 500 chain / 300 error-recovery)
- Recipe: LoRA r=128, alpha=256, lr=2e-4, max_steps=500, train_on_responses_only

### Colab usage
1. `Runtime → Change runtime type → T4 GPU`.
2. Upload `v7_2k/jarvis_v7_functiongemma.jsonl` to the Colab session (left panel → Files → upload). Keep the `v7_2k/` subfolder or update `DATASET_PATH` below to the flat filename.
3. `Runtime → Run all`.

Total run time on a free T4 is ~25-35 min for 500 steps.


## 1. Install Unsloth

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load the base model and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/functiongemma-270m-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128*2,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 3. Load the Jarvis dataset

Each row is `{"messages": "<JSON string>"}`. The JSON decodes to a list shaped like the TxT360 agent split: the first message carries a `tools` field, every assistant message has a `think` field, and tool responses carry `name` + `tool_call_id`. `prepare_messages_and_tools()` below does the exact same normalization the Unsloth recipe uses.

In [ ]:
from datasets import Dataset
import json, os

DATASET_PATH = "v7_2k/jarvis_v7_functiongemma.jsonl"
if not os.path.exists(DATASET_PATH):
    # fall back to flat-file upload (Colab single-file upload)
    DATASET_PATH = "jarvis_v7_functiongemma.jsonl"
assert os.path.exists(DATASET_PATH), (
    f"Upload `jarvis_v7_functiongemma.jsonl` to the Colab session first (left panel \u2192 Files \u2192 upload)."
)

rows = [json.loads(line) for line in open(DATASET_PATH)]
dataset = Dataset.from_list(rows)
print(f"Loaded {len(dataset)} examples from {DATASET_PATH}")
print("example messages (first 500 chars):", dataset[0]["messages"][:500])

## 4. Render with `apply_chat_template`

Same helper as the stock Unsloth notebook, verbatim: merges `think` into `content`, normalizes tool_calls, infers missing tool-response names, converts catalog-style tool schemas to HF format.

In [ ]:
import json

THINK_TAG_OPEN = "<think>"
THINK_TAG_CLOSE = "</think>"

def prepare_messages_and_tools(example):
    raw = json.loads(example["messages"])
    msgs = [dict(m) for m in raw]

    tools_raw = []
    if msgs and isinstance(msgs[0], dict):
        tlist = msgs[0].get("tools")
        if isinstance(tlist, list) and tlist:
            tools_raw = tlist
            msgs[0].pop("tools", None)

    THINK_KEYS = ["think", "think_fast", "think_faster"]
    has_valid_thought = False
    for m in msgs:
        if m.get("role") == "assistant":
            found_key = next((k for k in THINK_KEYS if m.get(k)), None)
            if found_key:
                think_text = m[found_key]
                content = m.get("content")
                think_block = f"{THINK_TAG_OPEN}{think_text}{THINK_TAG_CLOSE}"
                m["content"] = (think_block + "\n" + content) if isinstance(content, str) and content else think_block
                has_valid_thought = True
                for k in THINK_KEYS:
                    m.pop(k, None)
            else:
                return None, None
    if not has_valid_thought:
        return None, None

    for m in msgs:
        if "tool_calls" not in m or not m["tool_calls"]:
            continue
        new_tool_calls = []
        for tc in m["tool_calls"]:
            if not isinstance(tc, dict):
                continue
            if "function" in tc and isinstance(tc["function"], dict):
                new_tool_calls.append(tc)
                continue
            fn_name = tc.get("name", "")
            args = tc.get("arguments", {})
            if isinstance(args, str):
                try: args = json.loads(args)
                except Exception: pass
            new_tool_calls.append({
                "id": tc.get("id") or tc.get("tool_call_id"),
                "type": tc.get("type", "function"),
                "function": {"name": fn_name, "arguments": args},
            })
        m["tool_calls"] = new_tool_calls

    id_to_name = {}
    for m in msgs:
        for tc in m.get("tool_calls", []) or []:
            if not isinstance(tc, dict): continue
            fn = tc.get("function") or {}
            name = fn.get("name") or tc.get("name")
            tc_id = tc.get("id") or tc.get("tool_call_id")
            if tc_id and name: id_to_name[tc_id] = name

    for m in msgs:
        if m.get("role") == "tool" and not m.get("name"):
            tc_id = m.get("tool_call_id")
            m["name"] = id_to_name.get(tc_id) or "unknown_tool"

    adapted_tools = []
    for t in tools_raw:
        if not isinstance(t, dict): continue
        if "function" in t and isinstance(t["function"], dict):
            adapted_tools.append(t); continue
        adapted_tools.append({
            "type": t.get("type", "function"),
            "function": {
                "name": t.get("name", ""),
                "description": t.get("description", ""),
                "parameters": t.get("parameters") or {"type": "object", "properties": {}},
            },
        })

    if msgs and msgs[0].get("role") == "system" and "content" not in msgs[0]:
        msgs.pop(0)
    return msgs, adapted_tools

def format_example(example):
    messages, tools = prepare_messages_and_tools(example)
    if messages is None or len(messages) == 0:
        return {"text": None}
    chat_str = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=False, tokenize=False,
    ).removeprefix("<bos>")
    return {"text": chat_str}

train_dataset = dataset.map(format_example)
train_dataset = train_dataset.filter(lambda x: x["text"] is not None)
print(f"Dataset size after filtering: {len(train_dataset)}")

In [ ]:
# Sanity: render a single example to stdout so you can eyeball it.
print(train_dataset[0]["text"][:2500])

## 5. Train with SFTTrainer + train_on_responses_only

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        max_steps = 500,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for LoRA = {used_memory_for_lora} GB.")

## 6. Quick tool-call sanity check

In [ ]:
JARVIS_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "jm_list_pipelines",
            "description": "List all existing Jarvis pipelines under management",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_pipeline",
            "description": "Create a new Jarvis-CD pipeline environment",
            "parameters": {
                "type": "object",
                "properties": {"pipeline_id": {"type": "string", "description": "Name/ID for the pipeline"}},
                "required": ["pipeline_id"],
            },
        },
    },
]

SYSTEM = (
    "You are a Jarvis-CD HPC workflow assistant. Use the provided tools to "
    "create and manage pipelines, attach and configure packages, and operate "
    "the JarvisManager. Think briefly before each tool call."
)

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "List all my pipelines"},
]
text = tokenizer.apply_chat_template(
    messages, tools=JARVIS_TOOLS, tokenize=False, add_generation_prompt=True,
).removeprefix("<bos>")

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=256,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    top_p=0.95, top_k=64, temperature=1.0,
)

## 7. Save the LoRA adapter

In [ ]:
model.save_pretrained("jarvis_functiongemma_lora")
tokenizer.save_pretrained("jarvis_functiongemma_lora")

## 8. Export to GGUF for Ollama (optional)

Set the `if False:` gate to `if True:` when you want the full GGUF. Q8_0 is the standard quant used by this repo.

In [ ]:
if False:
    model.save_pretrained_merged("jarvis_functiongemma_16bit", tokenizer, save_method="merged_16bit")

if False:
    model.save_pretrained_gguf(
        "jarvis_functiongemma_gguf",
        tokenizer,
        quantization_method="Q8_0",
    )